# Paper 2 · Langkah 2: RQ1, kontrol format dan baseline metadata

Notebook ini menjawab bagian pertama RQ1: **bisakah kelas gambar (AI atau manusia) ditebak hanya dari data file, tanpa melihat isi gambarnya?**

Yang dikerjakan:
1. Membaca manifest dan menyalin 3.000 file ke disk Colab supaya cepat.
2. Mencatat ciri teknis JPEG tiap file (perkiraan kualitas kompresi, subsampling warna, progresif atau tidak).
3. Membuat pembagian 5-fold yang akan dipakai di semua langkah berikutnya.
4. Melatih model sederhana (regresi logistik) hanya pada metadata.
5. Menormalkan semua gambar: 256 × 256, JPEG kualitas 90, tanpa metadata.
6. Mengulang baseline pada file yang sudah dinormalkan.

Tidak butuh GPU. Jalankan dengan **Run all**. Perkiraan waktu: 5 sampai 10 menit.

## 1. Hubungkan Drive, baca manifest, salin file ke disk Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time
import numpy as np, pandas as pd
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Paper2')
RAW = PROJECT / 'data_asli' / 'ai_artbench_sample'
NORM = PROJECT / 'data_olahan' / 'ai_artbench_norm'
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
LOCAL = Path('/content/work')
for p in [NORM, RESULTS, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample.csv')
assert len(manifest) == 3000, f'Manifest berisi {len(manifest)} baris, seharusnya 3000.'

t0 = time.time()
local_raw = LOCAL / 'raw'
if not local_raw.exists():
    shutil.copytree(RAW, local_raw)
manifest['local_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)
missing = [p for p in manifest['local_path'] if not os.path.exists(p)]
assert not missing, f'{len(missing)} file tidak ditemukan, contoh: {missing[:3]}'
print(f'3.000 file siap di disk Colab ({time.time() - t0:.0f} detik).')

## 2. Ciri teknis JPEG per file

Ukuran file yang berbeda pada resolusi yang sama bisa berasal dari dua hal: isi gambar yang lebih sederhana, atau kompresi yang lebih kuat. Sel ini membaca tabel kuantisasi JPEG tiap file untuk memperkirakan kualitas kompresinya (skala 1 sampai 100), supaya dua hal itu bisa dipisahkan.

In [ ]:
from PIL import Image, JpegImagePlugin

# Tabel kuantisasi luminance standar (IJG) pada kualitas 50
STD_LUMA = np.array([
    16,11,10,16,24,40,51,61, 12,12,14,19,26,58,60,55, 14,13,16,24,40,57,69,56, 14,17,22,29,51,87,80,62,
    18,22,37,56,68,109,103,77, 24,35,55,64,81,104,113,92, 49,64,78,87,103,121,120,101, 72,92,95,98,112,100,103,99])

def jpeg_info(path):
    with Image.open(path) as im:
        q = getattr(im, 'quantization', None) or {}
        luma = np.array(q.get(0, []), dtype=float)
        if luma.size == 64:
            scale = 100.0 * luma.sum() / STD_LUMA.sum()
            quality = (200 - scale) / 2 if scale <= 100 else 5000 / scale
        else:
            quality = np.nan
        try:
            sub = JpegImagePlugin.get_sampling(im)
        except Exception:
            sub = -1
        return pd.Series({
            'jpeg_quality_est': round(float(quality), 1) if quality == quality else np.nan,
            'subsampling': {0: '4:4:4', 1: '4:2:2', 2: '4:2:0'}.get(sub, 'lain'),
            'progressive': int(bool(im.info.get('progressive', 0) or im.info.get('progression', 0))),
            'has_exif': int('exif' in im.info),
        })

manifest = pd.concat([manifest, manifest['local_path'].apply(jpeg_info)], axis=1)
manifest['file_size_kb'] = manifest['file_size_bytes'] / 1024

print('Perkiraan kualitas JPEG per sumber:')
display(manifest.groupby('source')['jpeg_quality_est'].describe().round(1))
print('\nSubsampling warna per sumber:')
display(manifest.groupby(['source', 'subsampling']).size().unstack(fill_value=0))
print('\nProgresif dan EXIF per sumber (jumlah file):')
display(manifest.groupby('source')[['progressive', 'has_exif']].sum())

## 3. Pembagian 5-fold

Fold dibuat sekali di sini dan disimpan, lalu dipakai ulang di semua langkah (baseline, detektor, peta atribusi). Fold diseimbangkan menurut kelas, generator, dan gaya.

In [ ]:
from sklearn.model_selection import StratifiedKFold

SEED = 42
strata = manifest['source'] + '|' + manifest['style']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
manifest['fold'] = -1
for k, (_, test_idx) in enumerate(skf.split(manifest, strata)):
    manifest.loc[test_idx, 'fold'] = k
assert (manifest['fold'] >= 0).all()
manifest[['sample_id', 'label', 'source', 'style', 'fold']].to_csv(NOTES / 'folds.csv', index=False)
print('Jumlah gambar per fold dan sumber:')
display(manifest.groupby(['fold', 'source']).size().unstack())

## 4. Baseline metadata pada file asli

Regresi logistik dilatih hanya pada data file. Tiga perbandingan: AI vs manusia (semua), manusia vs Latent Diffusion, manusia vs Stable Diffusion. Tiga set fitur: semua metadata, tanpa dimensi, dan ukuran file saja.

Cara membaca hasil: akurasi 0,50 berarti metadata tidak membantu sama sekali. Makin dekat ke 1,00, makin besar jalan pintas yang tersedia bagi detektor.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

def run_baseline(df, features, name, condition):
    X = pd.get_dummies(df[features], drop_first=False).astype(float)
    X = X.loc[:, X.nunique() > 1]            # buang kolom yang nilainya sama semua
    y = (df['label'] == 'ai').astype(int).values
    accs, aucs = [], []
    if X.shape[1] == 0:
        return {'kondisi': condition, 'perbandingan': name, 'fitur': ', '.join(features),
                'fitur_terpakai': '(tidak ada yang bervariasi)', 'akurasi': 0.5, 'akurasi_sd': 0.0,
                'auc': 0.5, 'auc_sd': 0.0, 'n': len(df)}
    for k in range(5):
        tr, te = (df['fold'] != k).values, (df['fold'] == k).values
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        model.fit(X[tr], y[tr])
        p = model.predict_proba(X[te])[:, 1]
        accs.append(accuracy_score(y[te], p >= 0.5)); aucs.append(roc_auc_score(y[te], p))
    return {'kondisi': condition, 'perbandingan': name, 'fitur': ', '.join(features),
            'fitur_terpakai': ', '.join(X.columns), 'akurasi': np.mean(accs), 'akurasi_sd': np.std(accs),
            'auc': np.mean(aucs), 'auc_sd': np.std(aucs), 'n': len(df)}

SUBSETS = {
    'AI vs manusia (semua)': manifest,
    'Manusia vs Latent Diffusion': manifest[manifest['source'].isin(['human', 'latent_diffusion'])],
    'Manusia vs Stable Diffusion': manifest[manifest['source'].isin(['human', 'stable_diffusion'])],
}
FEATURE_SETS = {
    'semua metadata': ['width', 'height', 'file_size_kb', 'jpeg_quality_est', 'subsampling', 'progressive', 'has_exif'],
    'tanpa dimensi': ['file_size_kb', 'jpeg_quality_est', 'subsampling', 'progressive', 'has_exif'],
    'ukuran file saja': ['file_size_kb'],
}
rows = []
for sname, sdf in SUBSETS.items():
    for fname, feats in FEATURE_SETS.items():
        r = run_baseline(sdf.reset_index(drop=True), feats, sname, 'file asli'); r['set_fitur'] = fname; rows.append(r)
res_raw = pd.DataFrame(rows)
display(res_raw[['perbandingan', 'set_fitur', 'akurasi', 'akurasi_sd', 'auc', 'auc_sd', 'n']].round(3))

## 5. Normalisasi

Setiap gambar dibuka, diubah ke RGB, dipotong persegi di tengah bila perlu, diubah ke 256 × 256 dengan interpolasi bicubic, lalu disimpan sekali sebagai JPEG kualitas 90 tanpa metadata. Subsampling warna dimatikan (4:4:4) supaya informasi warna tidak berkurang untuk analisis desain nanti.

File asli tidak disentuh. Hasil disimpan di `data_olahan/ai_artbench_norm/`.

In [ ]:
def normalise(src, dst):
    with Image.open(src) as im:
        im = im.convert('RGB')
        w, h = im.size
        s = min(w, h)
        crop_share = 1 - (s * s) / (w * h)
        if w != h:
            left, top = (w - s) // 2, (h - s) // 2
            im = im.crop((left, top, left + s, top + s))
        if im.size != (256, 256):
            im = im.resize((256, 256), Image.BICUBIC)
        dst.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst, 'JPEG', quality=90, subsampling=0, optimize=False)
    return crop_share

local_norm = LOCAL / 'norm'
t0 = time.time()
norm_paths, crops = [], []
for r in manifest.itertuples():
    dst = local_norm / r.source / r.style / f'{r.sample_id}.jpg'
    crops.append(normalise(r.local_path, dst)); norm_paths.append(str(dst))
manifest['norm_local_path'] = norm_paths
manifest['crop_share'] = crops
manifest['norm_file_size_kb'] = [os.path.getsize(p) / 1024 for p in norm_paths]
print(f'Normalisasi selesai ({time.time() - t0:.0f} detik). Luas terpotong maksimum: {max(crops):.3f}')

# Salin hasil ke Drive
if NORM.exists():
    shutil.rmtree(NORM)
shutil.copytree(local_norm, NORM)
manifest['norm_file_drive'] = manifest.apply(lambda r: str(NORM / r['source'] / r['style'] / f"{r['sample_id']}.jpg"), axis=1)
print('Tersimpan di', NORM)
print('\nUkuran file (KB) setelah normalisasi, per sumber:')
display(manifest.groupby('source')['norm_file_size_kb'].describe().round(1))

## 6. Baseline pada file yang sudah dinormalkan

Setelah normalisasi, format, dimensi, dan kompresi terakhir sudah sama untuk semua gambar. Satu-satunya metadata yang masih bervariasi adalah ukuran file, dan sekarang ukuran file mencerminkan **kerumitan isi gambar**, bukan pengaturan kompresi. Kalau ukuran file masih bisa menebak kelas, itu bukan lagi jalan pintas format, melainkan petunjuk bahwa gambar AI dan karya manusia berbeda dalam kerumitan visual.

In [ ]:
norm_df = manifest.assign(file_size_kb=manifest['norm_file_size_kb'])
rows = []
for sname in SUBSETS:
    sdf = norm_df.loc[SUBSETS[sname].index].reset_index(drop=True)
    r = run_baseline(sdf, ['file_size_kb'], sname, 'setelah normalisasi'); r['set_fitur'] = 'ukuran file saja'; rows.append(r)
res_norm = pd.DataFrame(rows)
display(res_norm[['perbandingan', 'set_fitur', 'akurasi', 'akurasi_sd', 'auc', 'auc_sd', 'n']].round(3))

## 7. Simpan hasil

In [ ]:
results = pd.concat([res_raw, res_norm], ignore_index=True)
results.to_csv(RESULTS / 'rq1_baseline_metadata.csv', index=False)
keep = [c for c in manifest.columns if c not in ('local_path', 'norm_local_path')]
manifest[keep].to_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv', index=False)

print('Tersimpan:')
print(' -', RESULTS / 'rq1_baseline_metadata.csv')
print(' -', NOTES / 'manifest_ai_artbench_sample_v2.csv')
print(' -', NOTES / 'folds.csv')
print('\nRINGKASAN RQ1 (baseline metadata)')
display(results[['kondisi', 'perbandingan', 'set_fitur', 'akurasi', 'auc']].round(3))

## Selesai

Setelah sel terakhir selesai, tekan **Cmd+S**, lalu kabari Claude. Catat juga tanggal dan hasil ringkasnya di log kerja.